# SIAM Conference on Computational Science and Engineering
`CSE19` https://meetings.siam.org/program.cfm?CONFCODE=CS19<br>
`CSE23` https://meetings.siam.org/program.cfm?CONFCODE=cse23<br>
`CSE25` https://meetings.siam.org/program.cfm?CONFCODE=cse25<br>

## Initialize

In [1]:
# ==== Step 0. Download NLTK Resources ====
import nltk
nltk.download("stopwords")
nltk.download("words")
nltk.download('punkt_tab')

!pip install gensim
!pip install transformers sentence-transformers

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/nanatsou/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package words to /Users/nanatsou/nltk_data...
[nltk_data]   Package words is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/nanatsou/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [2]:
# ==== Step 0. Import Libraries ====
import json, nltk, string, gensim, torch
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from nltk import FreqDist
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.metrics.pairwise import cosine_similarity

from collections import defaultdict
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster, inconsistent
from sentence_transformers import SentenceTransformer
from gensim.matutils import corpus2dense
from transformers import AutoTokenizer, AutoModel

## Read Conference Schedules

In [3]:
# ==== Step 0. Load JSON File ====
with open("CSE19_schd.json", "r") as f:
    schd19 = json.load(f)

with open("CSE23_schd.json", "r") as f:
    schd23 = json.load(f)

with open("CSE25_schd.json", "r") as f:
    schd25 = json.load(f)

In [4]:
# ==== Step 1. Collect information from the conference ====
"""
1. The real conference duration = len(days) + 1
2. In CSE19, "CP CSFD Submissions" is without talk
"""

def safe_join(items):
  return ' '.join(s for s in items if isinstance(s, str) and s.strip())

def analyze_conference_json(schd, conference_name):
  rooms = set()
  count_sessions = 0
  count_talks = 0
  days = set()
  timeslots = set()
  unique_authors = set()
  minisymposium_with_talks = 0
  cp_with_talks = 0
  session_type_counts = defaultdict(int)

  for session_id, session in schd.items():
      prefix = session_id[:2].upper()
      session_type_counts[prefix] += 1

      if "talks" in session and session["talks"]:
          count_sessions += 1
          count_talks += len(session["talks"])

          days.add(session.get("day"))
          rooms.add(session.get("room"))

          begin = session.get("begin_time")
          end = session.get("end_time")
          if begin and end:
              timeslots.add((session.get("day"), begin, end))

          if prefix == "MS":
              minisymposium_with_talks += 1
          if prefix == "CP":
              cp_with_talks += 1

          for talk in session["talks"]:
              if "authors" in talk:
                  for author in talk["authors"]:
                      name = author.get("name", "").strip()
                      affiliation = author.get("affiliation", "").strip()
                      email = (author.get("email") or "").strip().lower()
                      key = (name, affiliation, email)
                      unique_authors.add(key)

  df_type = pd.DataFrame(session_type_counts.items(), columns=["Session Type Prefix", "Count"])

  summary = {
      "Total number of MS & CP with talks": count_sessions,
      "Total number of MS with talks only": minisymposium_with_talks,
      "Total number of CP with talks only": cp_with_talks,
      "Total number of talks": count_talks,
      "Total number of unique days": len(days),
      "Total number of unique time slots": len(timeslots),
      "Total number of unique rooms": len(rooms),
      "Total number of unique authors": len(unique_authors),
  }
  df_summary = pd.DataFrame(list(summary.items()), columns = ["Metric", "Value"])

  print(f"==== Conference {conference_name} Summary ====")
  print(df_summary.to_string(index = False))
  print("\n==== Session Type Statistics ====")
  print(df_type.to_string(index = False))
  return rooms

analyze_conference_json(schd19, "CSE19")
analyze_conference_json(schd23, "CSE23")
analyze_conference_json(schd25, "CSE25");

==== Conference CSE19 Summary ====
                            Metric  Value
Total number of MS & CP with talks    421
Total number of MS with talks only    400
Total number of CP with talks only     21
             Total number of talks   1625
       Total number of unique days      5
 Total number of unique time slots     12
      Total number of unique rooms     38
    Total number of unique authors   2771

==== Session Type Statistics ====
Session Type Prefix  Count
                 IP      8
                 MS    400
                 CP     22
                 PD      6
                 SP      4
==== Conference CSE23 Summary ====
                            Metric  Value
Total number of MS & CP with talks    417
Total number of MS with talks only    417
Total number of CP with talks only      0
             Total number of talks   1754
       Total number of unique days      5
 Total number of unique time slots     14
      Total number of unique rooms     37
    Total number of

## Stage 1:  Text Representation and Embedding

## TF-IDF

In [5]:
# ==== Step 1. Clustering Parameters ====
# KEY CHANGE: the original NCLUSTERS = 20
rooms19 = analyze_conference_json(schd19, "CSE19")
rooms23 = analyze_conference_json(schd23, "CSE23")
rooms25 = analyze_conference_json(schd25, "CSE25")

NCLUSTERS_19 = len(rooms19)
NCLUSTERS_23 = len(rooms23)
NCLUSTERS_25 = len(rooms25)

# ==== Step 2. Set Common High-Frequency Words (24 words) & Acronyms (31 words) ====
COMMON_WORDS = ['advanc', 'activ', 'algorithm', 'applic', 'approach', 'challeng', 'confer',
                'develop', 'discuss', 'includ', 'method', 'minisymposium', 'new', 'numer',
                'problem', 'process', 'recent', 'research', 'session', 'solut', 'solver',
                'theori', 'use', 'workshop']

ACRONYMS = [
    ('AI', 'Artificial Intelligence'), ('AMR', 'Adaptive Mesh Refinement'), ('BEM', 'Block Element Modifier'),
    ('BGCE', 'Bavarian Graduate School of Computational Engineering'), ('CFD', 'Computational Fluid Dynamics'),
    ('CSE', 'Computational Science and Engineering'), ('DSL', 'Domain Specific Language'),
    ('FD', 'Finite Difference'), ('FEM', 'Finite Element Method'), ('FFT', 'Fast Fourier Transform'),
    ('GPU', 'Graphical Processing Unit'), ('HPC', 'High Performance Computing'),
    ('LGBT', "Lesbian Gay Bisexual and Transgender"), ('LGBTQ', "Lesbian Gay Bisequal Transgender and Queer"),
    ('MHD', 'Magnetohydrodynamics'), ('MISMC', 'Multi Index Sequential Monte Carlo'),
    ('ML', 'Machine Learning'), ('MOR', 'Mathematics of Operations Research'),
    ('NICAM', 'Near Instantaneous Companded Audio Multiplex'), ('NISQ', 'Noisy Intermediate Scale Quantum'),
    ('ODE', 'Ordinary Differential Equation'), ('OED', 'Optimal Experimental Design'),
    ('PDE', 'Partial Differential Equation'), ('RBF', 'Radial Basis Function'), ('RDM', 'Research Data Management'),
    ('ROM', 'Reduced Order Modelling'), ('RSE', 'Researchers Scientists and Engineers'),
    ('SBP', 'Summation By Parts'), ('SQP', 'Sequential Quadratic Programming'),
    ('SVD', 'Singular Value Decomposition'), ('UQ', 'Uncertainty Quantification')
]


def prepareTfidfEmbeddings(conference_prefix, schd):
  # ==== Step 3. KEY CHANGES - Simulate the Original minisymposia.yaml Structure ====
  mini_list = {}
  for session_code, session in schd.items():
      if "talks" in session and len(session["talks"]) > 0: # ensure it's symposium
          title = session["title"] if "title" in session else ""
          abstract = session["abstract"] if "abstract" in session else ""
          talk_titles = [t["title"] for t in session["talks"] if "title" in t]
          talk_abstracts = [t["abstract"] for t in session["talks"] if "abstract" in t]

          mini_list[session_code] = {
              "title": title,
              "abstract": abstract,
              "talk_titles": talk_titles,
              "talk_abstracts": talk_abstracts,
          }

  # ==== Step 4. Text Preprocessing ====

  # Set Stop words
  stop_words = set(stopwords.words("english") + COMMON_WORDS)

  # Create the stemmer
  stemmer = PorterStemmer()

  # KEY CHANGES # Hold the tokens for each title/abstract
  titles = list(mini_list.keys()) # use session code
  tokens = []

  for code in titles:
      ms_title = mini_list[code]["title"]
      ms_abstract = mini_list[code]["abstract"]
      talk_titles = mini_list[code]["talk_titles"]
      talk_abstracts = mini_list[code]["talk_abstracts"]


      # Combine title, abstract and talk information
      # content = f"{ms_title} {ms_title} {ms_abstract} {' '.join(talk_titles)} {' '.join(talk_abstracts)}"
      content = f"{ms_title} {ms_abstract} {safe_join(talk_titles)} {safe_join(talk_abstracts)}"

      # print(f"{code}: {len(content)} : {talk_abstracts}")

      # Replace acronyms
      for acro in ACRONYMS:
          content = content.replace(acro[0], acro[1])

      # Remove punctuation
      content = content.translate(str.maketrans(string.punctuation, ' ' * len(string.punctuation)))

      # Tokenize and lowercase
      words = word_tokenize(content.lower())

      # Remove stop words
      filtered = [word for word in words if word not in stop_words]

      # Stemming
      stemmed = [stemmer.stem(word) for word in filtered]

      # Final stop word removal (after stemming)
      filtered_final = [word for word in stemmed if word not in stop_words]
      tokens.append(filtered_final)

  # ==== Step 5. TF-IDF (Term Frequency-Inverse Document Frequency) Vectorization ====

  # Create a dictionary of words found in the program
  dictionary = gensim.corpora.Dictionary(tokens)

  # Compute the number of times each word appears in each abstract
  corpus = [dictionary.doc2bow(token) for token in tokens]

  # Lower the weight of common words
  tfidf = gensim.models.TfidfModel(corpus)

  # Create the similarity measure object
  sims = gensim.similarities.MatrixSimilarity(tfidf[corpus], num_features = len(dictionary))

  sims.save(f"{conference_prefix}tfidf_sims.data")
  print(f"Stored TFIDF MatrixSimilarity results to {conference_prefix+'sbert_embeddings.npy'}, sessions: {len(schd.items())}, ms:{len(mini_list)}, size: {len(sims)}")

==== Conference CSE19 Summary ====
                            Metric  Value
Total number of MS & CP with talks    421
Total number of MS with talks only    400
Total number of CP with talks only     21
             Total number of talks   1625
       Total number of unique days      5
 Total number of unique time slots     12
      Total number of unique rooms     38
    Total number of unique authors   2771

==== Session Type Statistics ====
Session Type Prefix  Count
                 IP      8
                 MS    400
                 CP     22
                 PD      6
                 SP      4
==== Conference CSE23 Summary ====
                            Metric  Value
Total number of MS & CP with talks    417
Total number of MS with talks only    417
Total number of CP with talks only      0
             Total number of talks   1754
       Total number of unique days      5
 Total number of unique time slots     14
      Total number of unique rooms     37
    Total number of

In [6]:
prepareTfidfEmbeddings("4_CSE19_", schd19)
prepareTfidfEmbeddings("4_CSE23_", schd23)
prepareTfidfEmbeddings("4_CSE25_", schd25)

Stored TFIDF MatrixSimilarity results to 4_CSE19_sbert_embeddings.npy, sessions: 440, ms:421, size: 421
Stored TFIDF MatrixSimilarity results to 4_CSE23_sbert_embeddings.npy, sessions: 439, ms:417, size: 417
Stored TFIDF MatrixSimilarity results to 4_CSE25_sbert_embeddings.npy, sessions: 345, ms:319, size: 319


## SBERT

In [7]:
def prepareSbertEmbeddings(conference_prefix, schd):
  # ==== Step 1. Preprocess for SBERT Embedding ====
  texts = []
  session_ids = []
  mini_list = {}

  # ==== Step 3. KEY CHANGES - Simulate the Original minisymposia.yaml Structure ====
  mini_list = {}
  for session_code, session in schd.items():
      if "talks" in session and len(session["talks"]) > 0: # ensure it's symposium
          title = session["title"] if "title" in session else ""
          abstract = session["abstract"] if "abstract" in session else ""
          talk_titles = [t["title"] for t in session["talks"] if "title" in t]
          talk_abstracts = [t["abstract"] for t in session["talks"] if "abstract" in t]

          mini_list[session_code] = {
              "title": title,
              "abstract": abstract,
              "talk_titles": talk_titles,
              "talk_abstracts": talk_abstracts,
          }
          content = f"{title} {abstract} {safe_join(talk_titles)} {safe_join(talk_abstracts)}"
          texts.append(content)
          session_ids.append(session_code)
  # ==== Step 2. Embedding using SBERT (mpnet) ====
  # reference: https://huggingface.co/sentence-transformers/all-mpnet-base-v2
  # map the textx above into a 768 dimensional dense vector space; the values are float number

  model = SentenceTransformer("all-mpnet-base-v2")
  sbert_embeddings = model.encode(texts, show_progress_bar = True)
  # Save to file
  np.save(conference_prefix+"sbert_embeddings.npy", sbert_embeddings)
  print(f"Stored embedding model results to {conference_prefix+'sbert_embeddings.npy'}, sessions: {len(schd.items())}, ms:{len(mini_list)}, size: {len(sbert_embeddings)}")

In [8]:
prepareSbertEmbeddings("4_CSE19_", schd19)
prepareSbertEmbeddings("4_CSE23_", schd23)
prepareSbertEmbeddings("4_CSE25_", schd25)

Batches:   0%|          | 0/14 [00:00<?, ?it/s]

Stored embedding model results to 4_CSE19_sbert_embeddings.npy, sessions: 440, ms:421, size: 421


Batches:   0%|          | 0/14 [00:00<?, ?it/s]

Stored embedding model results to 4_CSE23_sbert_embeddings.npy, sessions: 439, ms:417, size: 417


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

Stored embedding model results to 4_CSE25_sbert_embeddings.npy, sessions: 345, ms:319, size: 319


## SPECTER2

In [9]:
def prepareSpecter2Embeddings(conference_prefix, schd):
  # ==== Step 1. Preprocess for Specter 2 Embedding ====
  texts = []
  session_ids = []
  mini_list = {}


  def safe_join(items):
      return ' '.join(s for s in items if isinstance(s, str) and s.strip())
  # ==== Step 3. KEY CHANGES - Simulate the Original minisymposia.yaml Structure ====
  mini_list = {}
  for session_code, session in schd.items():
      if "talks" in session and len(session["talks"]) > 0: # ensure it's symposium
          ms_title = session["title"] if "title" in session else ""
          ms_abstract = session["abstract"] if "abstract" in session else ""
          talk_titles = [t["title"] for t in session["talks"] if "title" in t]
          talk_abstracts = [t["abstract"] for t in session["talks"] if "abstract" in t]

          mini_list[session_code] = {
              "title": ms_title,
              "abstract": ms_abstract,
              "talk_titles": talk_titles,
              "talk_abstracts": talk_abstracts,
          }
          content = f"{ms_title} {ms_abstract} {safe_join(talk_titles)} {safe_join(talk_abstracts)}"
          texts.append(content)
          session_ids.append(session_code)

  # ==== Step 2. Embedding using Specter2 via SentenceTransformer ====
  model = SentenceTransformer("allenai/specter2_base")
  specter_embeddings = model.encode(texts, batch_size = 16, show_progress_bar = True)
  np.save(conference_prefix + "specter_embeddings.npy", specter_embeddings)
  print(f"Stored embedding model results to {conference_prefix + 'specter_embeddings.npy'}, sessions: {len(schd.items())}, size: {len(specter_embeddings)}")

In [10]:
prepareSpecter2Embeddings("4_CSE19_", schd19)
prepareSpecter2Embeddings("4_CSE23_", schd23)
prepareSpecter2Embeddings("4_CSE25_", schd25)

No sentence-transformers model found with name allenai/specter2_base. Creating a new one with mean pooling.


Batches:   0%|          | 0/27 [00:00<?, ?it/s]

No sentence-transformers model found with name allenai/specter2_base. Creating a new one with mean pooling.


Stored embedding model results to 4_CSE19_specter_embeddings.npy, sessions: 440, size: 421


Batches:   0%|          | 0/27 [00:00<?, ?it/s]

No sentence-transformers model found with name allenai/specter2_base. Creating a new one with mean pooling.


Stored embedding model results to 4_CSE23_specter_embeddings.npy, sessions: 439, size: 417


Batches:   0%|          | 0/20 [00:00<?, ?it/s]

Stored embedding model results to 4_CSE25_specter_embeddings.npy, sessions: 345, size: 319
